In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

csv_path = "/kaggle/input/datasets/thedevastator/youtube-video-and-channel-analytics/YouTubeDataset_withChannelElapsed.csv"
df = pd.read_csv(csv_path)

In [7]:
# 1. Replace -1 sentinel values with 0 across the whole dataframe
df_clean = df.replace(-1, 0)

# 2. Ensure videoId is unique. Collapse duplicates by taking the first occurrence per videoId.
dup_count = df_clean.duplicated(subset='videoId').sum()
print(f"Duplicate videoId rows found: {dup_count}")

df_unique = df_clean.drop_duplicates(subset='videoId', keep='first').copy()
print(f"Rows before dedup: {len(df_clean)}, after dedup: {len(df_unique)}")

Duplicate videoId rows found: 19983
Rows before dedup: 575610, after dedup: 555627


In [9]:
# Question 1: Average number of likes per video

avg_likes_per_video = round(df_unique['videoLikeCount'].mean())
print(f"Average likes per video: {avg_likes_per_video}")

Average likes per video: 293


In [10]:
# Question 2: Average engagement rate across all videos
# engagement rate = (likes + dislikes + comments) / views, per video
# videos with 0 views: engagement rate = 0

engagement_numerator = (
    df_unique['videoLikeCount']
    + df_unique['videoDislikeCount']
    + df_unique['VideoCommentCount']
)

engagement_rate = np.where(
    df_unique['videoViewCount'] == 0,
    0,
    engagement_numerator / df_unique['videoViewCount']
)

avg_engagement_rate = round(engagement_rate.mean(), 4)
print(f"Average engagement rate: {avg_engagement_rate}")

Average engagement rate: 0.0084


In [11]:
# Question 3: Combined storage size per video in MiB

metadata_bytes = 500
thumbnail_bytes = 200 * 1024  # 1 KiB = 1024 bytes

total_bytes = metadata_bytes + thumbnail_bytes

total_mib = round(total_bytes / (1024 ** 2), 4)  # 1 MiB = 1024^2 bytes
print(f"Combined storage size per video: {total_mib} MiB")

Combined storage size per video: 0.1958 MiB


In [15]:
# Question 4: Total data transfer for all engagements 
# total engagements * per-video storage size

total_engagements = (
    df_unique['videoLikeCount']
    + df_unique['videoDislikeCount']
    + df_unique['VideoCommentCount']
).sum()

# unrounded combined storage size per video from Q3
total_mib_unrounded = total_bytes / (1024 ** 2)

total_data_transfer_mib = round(total_engagements * total_mib_unrounded)
print(f"Total data transfer: {total_data_transfer_mib} MiB")

Total data transfer: 38178142 MiB


In [17]:
# Question 5: Estimated views per minute over 5 years

total_views = df_unique['videoViewCount'].sum()

years = 5
days_per_year = 365
minutes_per_day = 24 * 60

total_minutes = years * days_per_year * minutes_per_day

views_per_minute = round(total_views / total_minutes)
print(f"Views per minute: {views_per_minute}")

Views per minute: 12473


In [18]:
# Question 6: Data accessed per minute

views_per_minute_unrounded = total_views / total_minutes

data_per_minute_mib = round(views_per_minute_unrounded * total_mib_unrounded)
print(f"Data accessed per minute: {data_per_minute_mib} MiB/minute")

Data accessed per minute: 2442 MiB/minute
